# 03 — Fixed-Income Basics: Bonds & FRA

This notebook covers:
- **Zero-coupon bonds**: NPV via discounting
- **Fixed-rate bonds**: clean/dirty price, accrued interest
- **Duration & convexity** via `jax.grad` (automatic differentiation)
- **Forward Rate Agreements** (FRA): pricing and settlement
- **vmap**: batch pricing across 1,000 yield scenarios

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare, compare_table, timed_ms, plot_speedup
jax = setup_backend(BACKEND)
import jax.numpy as jnp
QL = load_quantlib()

---
## 1. Zero-Coupon Bond

A zero-coupon bond pays face value $F$ at maturity $T$. Its present value:

$$
\text{NPV} = F \cdot P(0, T)
$$

In [ ]:
# ── QuantLib ────────────────────────────────────────────────────────────────
ql_today = QL.Date(15, 1, 2024)
QL.Settings.instance().evaluationDate = ql_today

ql_curve = QL.FlatForward(ql_today, 0.05, QL.Actual365Fixed())
ql_curve_handle = QL.YieldTermStructureHandle(ql_curve)

ql_zcb = QL.ZeroCouponBond(0, QL.TARGET(), 100.0,
                            QL.Date(15, 1, 2029))
ql_engine = QL.DiscountingBondEngine(ql_curve_handle)
ql_zcb.setPricingEngine(ql_engine)

print(f"QL ZCB NPV       : {ql_zcb.NPV():.10f}")
print(f"QL ZCB clean     : {ql_zcb.cleanPrice():.10f}")
print(f"QL ZCB dirty     : {ql_zcb.dirtyPrice():.10f}")

In [ ]:
# ── ql-jax ──────────────────────────────────────────────────────────────────
from ql_jax.time.date import Date
from ql_jax.time.calendar import NullCalendar, TARGET
from ql_jax.instruments.bond import make_zero_coupon_bond
from ql_jax.engines.bond.discounting import (
    discounting_bond_npv, discounting_bond_clean_price, discounting_bond_dirty_price
)
from ql_jax.termstructures.yield_.flat_forward import FlatForward

jax_today = Date(15, 1, 2024)
jax_curve = FlatForward(jax_today, 0.05)

jax_zcb = make_zero_coupon_bond(
    settlement_days=0, calendar=NullCalendar(),
    face_amount=100.0, maturity_date=Date(15, 1, 2029))

jax_npv   = float(discounting_bond_npv(jax_zcb, jax_curve))
jax_clean = float(discounting_bond_clean_price(jax_zcb, jax_curve))
jax_dirty = float(discounting_bond_dirty_price(jax_zcb, jax_curve))

print(f"JAX ZCB NPV       : {jax_npv:.10f}")
print(f"JAX ZCB clean     : {jax_clean:.10f}")
print(f"JAX ZCB dirty     : {jax_dirty:.10f}")

In [ ]:
compare_table([
    ("ZCB NPV",        ql_zcb.NPV(),        jax_npv),
    ("ZCB clean price", ql_zcb.cleanPrice(), jax_clean),
    ("ZCB dirty price", ql_zcb.dirtyPrice(), jax_dirty),
], title="Zero-Coupon Bond (5Y, 5% flat)")

---
## 2. Fixed-Rate Bond

A coupon bond with fixed periodic payments $c$ and face value $F$:

$$
\text{NPV} = \sum_{i=1}^{N} c \cdot F \cdot \tau_i \cdot P(0, t_i) + F \cdot P(0, T)
$$

We price three cases:
- **Par** bond (coupon = yield = 5%)
- **Discount** bond (coupon 3% < yield 5%)
- **Premium** bond (coupon 7% > yield 5%)

In [ ]:
# ── QuantLib ────────────────────────────────────────────────────────────────
ql_sched = QL.Schedule(
    ql_today, QL.Date(15, 1, 2029),
    QL.Period(QL.Annual), QL.NullCalendar(),
    QL.Unadjusted, QL.Unadjusted,
    QL.DateGeneration.Backward, False)

ql_bonds = {}
for label, cpn in [("Par 5%", 0.05), ("Discount 3%", 0.03), ("Premium 7%", 0.07)]:
    b = QL.FixedRateBond(0, 100.0, ql_sched, [cpn], QL.Actual365Fixed())
    b.setPricingEngine(ql_engine)
    ql_bonds[label] = b
    print(f"QL {label:14s}  NPV={b.NPV():10.6f}  clean={b.cleanPrice():10.6f}  dirty={b.dirtyPrice():10.6f}")

In [ ]:
# ── ql-jax ──────────────────────────────────────────────────────────────────
from ql_jax.time.schedule import MakeSchedule
from ql_jax._util.types import BusinessDayConvention, Frequency
from ql_jax.instruments.bond import make_fixed_rate_bond

jax_sched = (MakeSchedule()
    .from_date(jax_today)
    .to_date(Date(15, 1, 2029))
    .with_frequency(Frequency.Annual)
    .with_calendar(NullCalendar())
    .with_convention(BusinessDayConvention.Unadjusted)
    .build())

jax_bonds = {}
for label, cpn in [("Par 5%", 0.05), ("Discount 3%", 0.03), ("Premium 7%", 0.07)]:
    b = make_fixed_rate_bond(
        settlement_days=0, face_amount=100.0,
        schedule=jax_sched, coupons=cpn,
        day_counter="Actual/365 (Fixed)")
    npv_   = float(discounting_bond_npv(b, jax_curve))
    clean_ = float(discounting_bond_clean_price(b, jax_curve))
    dirty_ = float(discounting_bond_dirty_price(b, jax_curve))
    jax_bonds[label] = (b, npv_, clean_, dirty_)
    print(f"JAX {label:14s}  NPV={npv_:10.6f}  clean={clean_:10.6f}  dirty={dirty_:10.6f}")

In [ ]:
rows = []
for label in ["Par 5%", "Discount 3%", "Premium 7%"]:
    ql_b = ql_bonds[label]
    _, jnpv, jclean, jdirty = jax_bonds[label]
    rows.append((f"{label} NPV",   ql_b.NPV(),        jnpv))
    rows.append((f"{label} clean", ql_b.cleanPrice(),  jclean))
    rows.append((f"{label} dirty", ql_b.dirtyPrice(),  jdirty))

compare_table(rows, title="Fixed-Rate Bonds (5Y Annual, 5% flat curve)")

---
## 3. Duration & Convexity via AD

**Modified duration** is the first-order price sensitivity to yield:

$$
D_{\text{mod}} = -\frac{1}{P} \frac{dP}{dy}
$$

**Convexity** is the second-order sensitivity:

$$
C = \frac{1}{P} \frac{d^2P}{dy^2}
$$

With JAX, we compute these **exactly via automatic differentiation** — no finite-difference bumping required.

In [ ]:
# Bond price as a function of yield (for AD)
par_bond_jax = jax_bonds["Par 5%"][0]

def bond_price_fn(y):
    """Clean price of the par bond given flat yield y."""
    curve = FlatForward(jax_today, y)
    return discounting_bond_clean_price(par_bond_jax, curve)

y0 = jnp.float64(0.05)
P0 = bond_price_fn(y0)

# First derivative → duration
dP_dy = jax.grad(bond_price_fn)(y0)
mod_duration_ad = -float(dP_dy) / float(P0)

# Second derivative → convexity
d2P_dy2 = jax.grad(jax.grad(bond_price_fn))(y0)
convexity_ad = float(d2P_dy2) / float(P0)

print(f"Bond price     : {float(P0):.6f}")
print(f"dP/dy (AD)     : {float(dP_dy):.6f}")
print(f"Mod duration   : {mod_duration_ad:.6f}")
print(f"d²P/dy² (AD)   : {float(d2P_dy2):.6f}")
print(f"Convexity      : {convexity_ad:.6f}")

In [ ]:
# Compare to QuantLib's BondFunctions.duration
ql_par = ql_bonds["Par 5%"]
ql_mod_dur = QL.BondFunctions.duration(
    ql_par, QL.InterestRate(0.05, QL.Actual365Fixed(), QL.Continuous, QL.Annual),
    QL.Duration.Modified)

print(f"\nQL  Modified Duration : {ql_mod_dur:.6f}")
print(f"JAX Modified Duration : {mod_duration_ad:.6f}")
print(f"Difference            : {abs(ql_mod_dur - mod_duration_ad):.2e}")

In [ ]:
# Visualize: price-yield curve with tangent line
import matplotlib.pyplot as plt
import numpy as np

yields = np.linspace(0.01, 0.10, 100)
prices = [float(bond_price_fn(jnp.float64(y))) for y in yields]

# Tangent at 5%
tangent = float(P0) + float(dP_dy) * (yields - 0.05)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(yields * 100, prices, 'b-', linewidth=2, label='Price(yield)')
ax.plot(yields * 100, tangent, 'r--', linewidth=1, label='Tangent (duration)')
ax.plot(5.0, float(P0), 'ko', markersize=8)
ax.set_xlabel('Yield (%)')
ax.set_ylabel('Clean Price')
ax.set_title('Bond Price-Yield Curve with Duration Tangent')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 4. Forward Rate Agreement (FRA)

A FRA is a contract to lock in a future borrowing/lending rate. The payoff at settlement:

$$
\text{Payoff} = \frac{N \cdot \tau \cdot (R - K)}{1 + \tau \cdot R}
$$

where $R$ is the realized rate, $K$ is the strike, $\tau$ is the day-count fraction.

In [ ]:
# ── QuantLib FRA ───────────────────────────────────────────────────────────
ql_fra = QL.ForwardRateAgreement(
    ql_today + QL.Period(3, QL.Months),
    ql_today + QL.Period(9, QL.Months),
    QL.Position.Long,
    0.045,   # strike
    1_000_000.0,
    QL.Euribor6M(ql_curve_handle))

print(f"QL FRA forward rate : {ql_fra.forwardRate():.10f}")
print(f"QL FRA NPV          : {ql_fra.NPV():.6f}")

In [ ]:
# ── ql-jax FRA ─────────────────────────────────────────────────────────────
from ql_jax.instruments.fra import ForwardRateAgreement, FRAType
from ql_jax._util.types import TimeUnit

jax_fra = ForwardRateAgreement(
    value_date=jax_today.advance(3, TimeUnit.Months),
    maturity_date=jax_today.advance(9, TimeUnit.Months),
    strike=0.045,
    notional=1_000_000.0,
    type_=FRAType.Long,
)

jax_fra_fwd = float(jax_fra.forward_rate(jax_curve))
jax_fra_npv = float(jax_fra.npv(jax_curve))

print(f"JAX FRA forward rate : {jax_fra_fwd:.10f}")
print(f"JAX FRA NPV          : {jax_fra_npv:.6f}")

In [ ]:
compare_table([
    ("FRA forward rate", ql_fra.forwardRate(), jax_fra_fwd),
    ("FRA NPV",          ql_fra.NPV(),         jax_fra_npv),
], title="FRA 3×9 (strike 4.5%, 5% flat curve)")

---
## 5. JAX-Unique: Batch Bond Pricing via vmap

Price the par bond across 1,000 different yield scenarios in a **single vectorized call**.

In [ ]:
# Vectorized bond pricing
def bond_clean_from_yield(y):
    curve = FlatForward(jax_today, y)
    return discounting_bond_clean_price(par_bond_jax, curve)

# 1000 yield scenarios: 1% to 10%
yield_scenarios = jnp.linspace(0.01, 0.10, 1000)

# vmap for batch pricing
batch_price = jax.vmap(bond_clean_from_yield)(yield_scenarios)

# Also compute duration for each scenario
batch_duration = jax.vmap(jax.grad(bond_clean_from_yield))(yield_scenarios)

print(f"Computed {len(batch_price)} prices and durations in one call")
print(f"Price range: [{float(batch_price.min()):.2f}, {float(batch_price.max()):.2f}]")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(np.array(yield_scenarios) * 100, np.array(batch_price), 'b-')
ax1.set_xlabel('Yield (%)')
ax1.set_ylabel('Clean Price')
ax1.set_title('1,000 Bond Prices via vmap')
ax1.grid(True, alpha=0.3)

ax2.plot(np.array(yield_scenarios) * 100, -np.array(batch_duration) / np.array(batch_price), 'r-')
ax2.set_xlabel('Yield (%)')
ax2.set_ylabel('Modified Duration')
ax2.set_title('Duration vs Yield (via jax.grad + vmap)')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## 6. Performance: Bond Pricing

In [ ]:
def ql_price_bond():
    return ql_bonds["Par 5%"].NPV()

def jax_price_bond():
    return float(discounting_bond_npv(par_bond_jax, jax_curve))

# JIT-compile the jax version
jit_bond_npv = jax.jit(lambda: discounting_bond_npv(par_bond_jax, jax_curve))
_ = jit_bond_npv()  # warmup

ql_ms, _  = timed_ms(ql_price_bond, warmup=5, runs=20)
jax_ms, _ = timed_ms(jax_price_bond, warmup=5, runs=20)
jit_ms, _ = timed_ms(lambda: float(jit_bond_npv()), warmup=5, runs=20)

print(f"Bond NPV (single):")
print(f"  QuantLib   : {ql_ms:.4f} ms")
print(f"  ql-jax     : {jax_ms:.4f} ms")
print(f"  ql-jax JIT : {jit_ms:.4f} ms")

plot_speedup(
    ["NPV", "NPV (JIT)"],
    [ql_ms, ql_ms],
    [jax_ms, jit_ms],
    title="Bond NPV: QuantLib vs ql-jax"
)

---
## 7. Exercises

1. **Yield-to-maturity**: Use `jax.scipy.optimize` (or a simple root-finding loop) to compute the yield of the discount bond given its price. Compare to QuantLib's `BondFunctions.yield_`.
2. **Key-rate duration**: Compute $\partial P / \partial z_i$ for each zero rate pillar using `jax.jacrev` through a piecewise curve.
3. **FRA sensitivity**: Use `jax.grad` to compute dNPV/dRate for the FRA and verify against a 1bp finite-difference bump.

---
*End of Notebook 03*